In [1]:
import gc
import numpy as np
import polars as pl
import json
import pandas as pd

In [2]:
!pip install polars

In [3]:
try:
    import pyarrow
    print("PyArrow is installed. Version:", pyarrow.__version__)
except ModuleNotFoundError:
    print("PyArrow is NOT found in this environment. You installed it in a different one.")

PyArrow is installed. Version: 23.0.1


In [4]:
train = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/train_main_features.parquet')
test = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/test_main_features.parquet')

In [5]:
test_extra = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/test_extra_features.parquet')

In [6]:
feature_importance_rf = pd.read_json('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/feature_importance_rf.json', orient='records')
print("Загружено из JSON:")
print(feature_importance_rf.head())
feature_importance_cat = pd.read_json('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/feature_importance_cat.json', orient='records')
print("Загружено из JSON:")
print(feature_importance_cat.head())

Загружено из JSON:
            feature  importance_rf
0    num_feature_87       0.004691
1    num_feature_62       0.003942
2    num_feature_27       0.003800
3   num_feature_879       0.003702
4  num_feature_1377       0.003294
Загружено из JSON:
            feature  importance_cat
0    num_feature_41        3.462705
1    num_feature_22        1.337958
2  num_feature_1309        1.239737
3  num_feature_1644        1.128667
4    num_feature_42        1.092923


In [7]:
# Объединяем важности из двух источников
combined_importance = feature_importance_cat.copy()
combined_importance = combined_importance.rename(columns={'importance': 'importance_cat'})
combined_importance['importance_rf'] = feature_importance_rf['importance_rf']

combined_importance['importance_cat_norm'] = (
    combined_importance['importance_cat'] / combined_importance['importance_cat'].max()
)
combined_importance['importance_rf_norm'] = (
    combined_importance['importance_rf'] / combined_importance['importance_rf'].max()
)

combined_importance['importance_combined'] = (
    combined_importance['importance_cat_norm'] * 0.6 +
    combined_importance['importance_rf_norm'] * 0.4
)

combined_importance = combined_importance.sort_values('importance_combined', ascending=False)
combined_importance['cumulative_importance'] = combined_importance['importance_combined'].cumsum()
combined_importance['cumulative_pct'] = combined_importance['cumulative_importance'] / combined_importance['importance_combined'].sum()

In [8]:
#  Выбираем топ-1600
top_features = feature_importance_cat.head(1600)['feature'].tolist()

In [9]:
top_main_features = [f for f in top_features if f in train.columns]
top_extra_features = [f for f in top_features if f in test_extra.columns]

print(f"\nВыбрано признаков: {len(top_features)}")
print(f"  Из основного набора: {len(top_main_features)}")
print(f"  Из дополнительного: {len(top_extra_features)}")


Выбрано признаков: 1600
  Из основного набора: 149
  Из дополнительного: 1451


In [10]:
train_main = train[['customer_id'] + top_main_features]
train_main.head()

customer_id,num_feature_41,num_feature_22,num_feature_42,num_feature_117,num_feature_76,cat_feature_8,num_feature_27,num_feature_126,num_feature_71,num_feature_87,num_feature_62,cat_feature_12,num_feature_67,num_feature_63,num_feature_16,num_feature_73,num_feature_70,num_feature_104,num_feature_46,cat_feature_40,num_feature_68,num_feature_99,num_feature_85,num_feature_116,cat_feature_20,num_feature_52,cat_feature_60,num_feature_35,cat_feature_17,num_feature_88,num_feature_24,num_feature_23,cat_feature_49,num_feature_7,num_feature_30,num_feature_25,…,num_feature_82,num_feature_91,num_feature_3,cat_feature_9,num_feature_4,num_feature_110,num_feature_8,num_feature_102,cat_feature_67,num_feature_48,num_feature_65,num_feature_66,num_feature_12,cat_feature_25,cat_feature_36,num_feature_2,cat_feature_48,cat_feature_32,num_feature_45,cat_feature_13,num_feature_49,num_feature_79,cat_feature_28,cat_feature_45,num_feature_96,num_feature_13,num_feature_34,num_feature_124,num_feature_127,cat_feature_43,cat_feature_50,num_feature_94,num_feature_53,cat_feature_2,num_feature_78,cat_feature_66,cat_feature_5
i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1000001,0.445712,null,-0.002153,-0.019079,-0.027542,2.0,-0.020738,-0.445279,null,null,-0.02543,0.0,-0.430568,null,null,-0.341678,null,0.450342,-0.008472,0.0,-0.363338,null,null,-0.493959,0.0,null,2.0,0.453232,2.0,null,-0.542784,null,0.0,-0.1476,null,null,…,-0.028607,null,-0.157258,4.0,null,null,-0.257746,-0.043592,2.0,1.61104,-0.290711,0.01146,-0.102245,0.0,0.0,null,0.0,0.0,null,0.0,-0.001544,-0.040404,2.0,0.0,-0.284519,null,null,-0.046146,null,2.0,1.0,-0.126543,-0.094058,0.0,-0.097437,2.0,2.0
1000002,-0.583389,null,-0.001728,-0.014154,-0.027717,0.0,-0.004672,1.550722,null,-0.018017,-0.023054,0.0,-0.430568,null,-0.005735,-0.341678,null,1.998252,-0.013828,1.0,1.179635,null,2.330621,-0.256445,1.0,-0.066879,0.0,0.346346,0.0,null,-0.571248,null,0.0,4.08256,-0.011896,null,…,-0.028607,null,-0.286426,0.0,null,null,-0.257746,-0.043592,0.0,-0.139625,-0.290711,null,-0.102245,0.0,0.0,-0.210019,0.0,1.0,null,0.0,-0.001544,-0.040404,0.0,1.0,-0.284519,null,null,-0.046146,null,0.0,0.0,1.167873,-0.094058,0.0,-0.097437,0.0,0.0
1000003,-0.197476,null,null,-0.019124,-0.02765,0.0,-0.020788,-0.475778,null,null,-0.025546,0.0,-0.430568,null,null,-0.341678,0.0,-0.264078,-0.032381,3.0,-0.591927,null,null,-0.57313,0.0,null,0.0,0.427056,0.0,null,-0.542784,null,0.0,-0.1476,-0.01193,null,…,-0.028607,null,null,4.0,null,null,-0.257746,-0.043592,0.0,-0.139625,-0.290711,null,-0.102245,0.0,0.0,-0.210019,0.0,0.0,null,0.0,-0.001544,-0.040404,0.0,0.0,-0.284519,null,null,-0.046146,null,0.0,0.0,-0.234411,-0.094058,0.0,-0.097437,2.0,0.0
1000004,null,null,null,null,null,2.0,null,-0.475778,null,null,null,0.0,-0.430568,null,-0.005735,-0.341678,0.0,0.688482,-0.02422,3.0,-0.591927,null,-0.104497,-0.57313,0.0,null,2.0,-0.070289,2.0,null,-0.542784,null,0.0,-0.1476,null,null,…,null,0.008037,null,3.0,null,null,-0.257746,-0.043592,2.0,-0.139625,-0.290711,-0.28984,-0.102245,0.0,0.0,null,1.0,0.0,null,0.0,-0.001544,-0.040404,2.0,2.0,-0.284519,null,null,-0.046146,0.111196,2.0,1.0,-0.342279,-0.094058,0.0,-0.097437,0.0,2.0
1000005,-0.004519,null,null,-0.018674,-0.027545,0.0,-0.019913,null,null,null,-0.025267,2.0,1.140868,-0.544196,null,-0.341678,null,-0.264078,0.060004,3.0,-0.591927,null,-0.155999,-0.57313,2.0,null,0.0,0.553573,0.0,null,null,null,2.0,-0.1476,null,null,…,-0.028607,null,null,2.0,null,null,1.418979,-0.043592,0.0,null,-0.290711,null,null,2.0,2.0,-0.210019,2.0,2.0,null,2.0,null,-0.040404,0.0,0.0,-0.284519,null,null,-0.046146,null,0.0,0.0,null,null,2.0,-0.097437,2.0,0.0


In [11]:
test_main = test[['customer_id'] + top_main_features]
test_main.head()

customer_id,num_feature_41,num_feature_22,num_feature_42,num_feature_117,num_feature_76,cat_feature_8,num_feature_27,num_feature_126,num_feature_71,num_feature_87,num_feature_62,cat_feature_12,num_feature_67,num_feature_63,num_feature_16,num_feature_73,num_feature_70,num_feature_104,num_feature_46,cat_feature_40,num_feature_68,num_feature_99,num_feature_85,num_feature_116,cat_feature_20,num_feature_52,cat_feature_60,num_feature_35,cat_feature_17,num_feature_88,num_feature_24,num_feature_23,cat_feature_49,num_feature_7,num_feature_30,num_feature_25,…,num_feature_82,num_feature_91,num_feature_3,cat_feature_9,num_feature_4,num_feature_110,num_feature_8,num_feature_102,cat_feature_67,num_feature_48,num_feature_65,num_feature_66,num_feature_12,cat_feature_25,cat_feature_36,num_feature_2,cat_feature_48,cat_feature_32,num_feature_45,cat_feature_13,num_feature_49,num_feature_79,cat_feature_28,cat_feature_45,num_feature_96,num_feature_13,num_feature_34,num_feature_124,num_feature_127,cat_feature_43,cat_feature_50,num_feature_94,num_feature_53,cat_feature_2,num_feature_78,cat_feature_66,cat_feature_5
i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1750001,0.960263,null,null,-0.01556,-0.027352,2.0,-0.018283,-0.475778,null,null,-0.024379,0.0,-0.430568,null,-0.005735,-0.341678,null,-0.383148,-0.023483,0.0,0.036692,-0.150243,null,-0.493959,0.0,-0.066879,2.0,-0.227345,2.0,null,-0.542784,null,0.0,-0.1476,null,null,…,-0.028607,null,null,4.0,null,null,-0.257746,-0.043592,2.0,-0.139625,-0.290711,null,-0.102245,0.0,0.0,-0.210019,0.0,0.0,null,0.0,-0.001544,-0.040404,2.0,0.0,0.503368,null,null,-0.046146,null,2.0,1.0,0.089193,-0.094058,0.0,-0.097437,0.0,2.0
1750002,-0.712026,null,null,-0.015727,-0.026929,2.0,-0.016736,-0.475778,null,null,-0.024826,0.0,-0.430568,null,null,-0.341678,null,-0.025938,-0.018151,0.0,-0.363338,0.386748,null,-0.57313,1.0,null,2.0,0.954938,2.0,null,-0.514319,null,1.0,-0.1476,-0.01193,null,…,-0.028607,null,null,4.0,null,null,-0.257746,-0.043592,2.0,2.8312,-0.290711,null,-0.102245,0.0,0.0,-0.210019,0.0,1.0,null,0.0,-0.001544,-0.040404,2.0,0.0,-0.284519,null,null,-0.046146,null,2.0,1.0,0.844269,-0.094058,0.0,-0.097437,2.0,2.0
1750003,1.153219,null,null,-0.01934,-0.027116,2.0,-0.020352,-0.455446,null,null,-0.025318,0.0,-0.430568,-0.035788,-0.005735,null,null,-0.264078,-0.009617,3.0,null,null,null,null,0.0,-0.066879,2.0,null,2.0,null,-0.229675,null,0.0,null,null,null,…,-0.028607,null,null,4.0,null,null,-0.257746,null,2.0,-0.139625,null,null,-0.102245,0.0,0.0,null,0.0,1.0,null,0.0,-0.001544,null,2.0,0.0,null,null,null,null,null,2.0,1.0,-0.342279,-0.094058,0.0,null,0.0,2.0
1750004,0.381394,null,null,-0.015257,-0.027451,2.0,-0.018194,-0.475778,null,null,-0.025244,0.0,-0.430568,null,null,0.934265,null,-0.383148,-0.04039,0.0,0.265281,-0.150243,null,-0.018932,0.0,null,2.0,-0.521825,2.0,null,-0.542784,null,0.0,-0.1476,null,null,…,null,null,null,4.0,null,null,-0.257746,-0.043592,2.0,-0.139625,2.991479,-0.55795,-0.102245,0.0,0.0,null,0.0,0.0,null,0.0,-0.001544,-0.040404,2.0,0.0,-0.284519,null,null,-0.046146,null,2.0,1.0,-0.342279,-0.094058,0.0,-0.097437,2.0,2.0
1750005,null,null,null,-0.017804,-0.027126,2.0,-0.017956,1.533778,null,null,-0.024389,0.0,-0.430568,null,null,-0.341678,null,-0.383148,-0.023725,3.0,-0.020455,-0.150243,null,-0.57313,1.0,null,2.0,0.322352,2.0,null,-0.514319,null,1.0,-0.1476,null,null,…,-0.028607,-0.079564,-0.255161,4.0,null,null,-0.257746,-0.043592,2.0,-0.139625,-0.290711,-0.388693,-0.102245,1.0,0.0,null,0.0,1.0,null,0.0,-0.001544,-0.040404,2.0,0.0,-0.284519,null,null,-0.046146,1.318607,2.0,1.0,-0.342279,-0.094058,1.0,-0.097437,2.0,2.0


In [12]:
lazy_extra = pl.scan_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/train_extra_features.parquet')

lazy_extra = lazy_extra.cast(pl.Float64)

lazy_extra = lazy_extra.select(['customer_id'] + top_extra_features)

train_extra = lazy_extra.collect()

In [13]:
train_extra = train_extra.with_columns(
    [pl.col(c).cast(pl.Float32) for c in train_extra.columns if train_extra[c].dtype == pl.Float64]
)
train_extra = train_extra.with_columns(
    pl.col('customer_id').cast(pl.Int32)
)

In [14]:
test_extra = test_extra[['customer_id'] + top_extra_features]

In [15]:
test_extra = test_extra.with_columns(
    [pl.col(c).cast(pl.Float32) for c in train_extra.columns if test_extra[c].dtype == pl.Float64]
)
test_extra = test_extra.with_columns(
    pl.col('customer_id').cast(pl.Int32)
)

In [16]:
train_combined = train_main.join(
    train_extra,
    on='customer_id',
    how='left'  
)

print("Размер после объединения train и extra:", train_combined.height)
print("Количество столбцов:", train_combined.width)

train_combined.write_parquet('train_combined_1600.parquet')

Размер после объединения train и extra: 750000
Количество столбцов: 1601


In [17]:
test_combined = test_main.join(
    test_extra,
    on='customer_id',
    how='left'  
)

print("Размер после объединения test и extra:", test_combined.height)
print("Количество столбцов:", test_combined.width)

test_combined.write_parquet('test_combined_1600.parquet')

Размер после объединения test и extra: 250000
Количество столбцов: 1601
